# Kaza Phase 1 — Step 1: Colab Pro Setup

In [ ]:
!test -d /content/kaza && git -C /content/kaza pull --ff-only origin phase1-design || git clone -b phase1-design https://github.com/udohchuks/kaza.git /content/kaza
%cd /content/kaza
!pip install -q -r requirements.txt

In [ ]:
from kaza.env import print_env

env = print_env()

In [ ]:
import gc, torch
from transformers import AutoModel, AutoModelForCausalLM

codec = AutoModel.from_pretrained("LiquidAI/Liquid-Codec", trust_remote_code=True).to(env.device).eval()
print(f"Liquid-Codec: {sum(p.numel() for p in codec.parameters()) / 1e6:.1f}M params")
del codec; gc.collect(); torch.cuda.empty_cache()

backbone = AutoModelForCausalLM.from_pretrained("LiquidAI/LFM3-1.7B-Base", dtype=env.frozen_dtype).to(env.device).eval()
with torch.no_grad(), torch.amp.autocast(env.device.type, dtype=env.autocast_dtype):
    logits = backbone(input_ids=torch.zeros((1, 8), dtype=torch.long, device=env.device)).logits
print(f"LFM3-1.7B-Base: {len(backbone.model.layers)} layers | logits {tuple(logits.shape)} | finite={torch.isfinite(logits).all().item()}")
del backbone, logits; gc.collect(); torch.cuda.empty_cache()